# Agentic RAG - Asisten Riset Oftalmologi

Notebook eksperimen untuk memvalidasi pipeline RAG dasar (loader -> splitter -> embedding -> vector store -> chain tanya jawab) menggunakan paper riset sendiri sebagai dokumen pertama.

Tahap ini murni untuk validasi pendekatan di Colab. Setelah retrieval terbukti akurat, kode dipindahkan ke struktur project FastAPI + LangGraph.

## 1. Install dependency

In [1]:
!pip install -qU langchain langchain-community langchain-google-genai langchain-chroma langchain-text-splitters pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 834.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.7/164.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262

## 2. Setup API key (Colab Secrets)

Klik ikon kunci di sidebar kiri Colab, tambahkan secret dengan nama `GOOGLE_API_KEY`, aktifkan akses notebook untuk secret tersebut, baru jalankan cell di bawah.

In [2]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

## 3. Upload paper (PDF)

Pilih file PDF paper riset kamu (JAIC) saat prompt upload muncul.

In [3]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f"File terupload: {pdf_path}")

Saving Azrul_1835-1845 (2).pdf to Azrul_1835-1845 (2).pdf
File terupload: Azrul_1835-1845 (2).pdf


## 4. (Opsional) Mount Google Drive

Jalankan cell ini kalau ingin vector store tersimpan permanen antar sesi Colab. Kalau cuma eksperimen sekali jalan, lewati saja cell ini dan biarkan `persist_directory` di langkah 7 memakai path lokal Colab (`./chroma_db`), yang akan hilang saat runtime disconnect.

In [4]:
# from google.colab import drive
# drive.mount('/content/drive')
# PERSIST_DIR = "/content/drive/MyDrive/langchain-rag-project/chroma_db"

PERSIST_DIR = "./chroma_db"  # ganti ke path Drive di atas kalau mount Drive

## 5. Load dan split dokumen

In [5]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(pdf_path)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " "]
)
chunks = splitter.split_documents(documents)

print(f"Jumlah halaman: {len(documents)}")
print(f"Jumlah chunk: {len(chunks)}")
print("\n Contoh chunk pertama ")
print(chunks[0].page_content[:500])

/tmp/ipykernel_1479/3519770123.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Jumlah halaman: 11
Jumlah chunk: 69

 Contoh chunk pertama 
Journal of Applied Informatics and Computing (JAIC) 
Vol.9, No.4, August 2025, pp. 1835~1845 
e-ISSN: 2548-6861    1835 
  
  
http://jurnal.polibatam.ac.id/index.php/JAIC  
Enhancing Eye Diseases Classification Using Imbalance Training & 
Machine Learning 
 
 
Muhammad Azrul Ihwan 1*, Ajie Kusuma Wardhana 2* 
* Informatika, Universitas Amikom Yogyakarta 
muhammadazrulihwan@students.amikom.ac.id 1, ajiekusuma@amikom.ac.id  2,  
 
 
Article Info  ABSTRACT  
Article history: 
Received 2025-07-13 



## 6. Tambahkan metadata sitasi

Sesuaikan nilai judul, tahun, dan penulis dengan paper kamu.

In [6]:
SOURCE_TITLE = "Machine Learning Algorithms for Eye Disease Classification"
SOURCE_YEAR = 2025
SOURCE_AUTHOR = "Azrul, dkk"

for chunk in chunks:
    chunk.metadata["source_title"] = SOURCE_TITLE
    chunk.metadata["year"] = SOURCE_YEAR
    chunk.metadata["author"] = SOURCE_AUTHOR

print(chunks[0].metadata)

{'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2025-08-08T19:48:08+07:00', 'moddate': '2025-08-08T19:48:25+07:00', 'title': '', 'author': 'Azrul, dkk', 'subject': '', 'keywords': '', 'source': 'Azrul_1835-1845 (2).pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_title': 'Machine Learning Algorithms for Eye Disease Classification', 'year': 2025}


## 7. Buat embedding dan simpan ke Chroma

In [8]:
import google.generativeai as genai
import os

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

for m in genai.list_models():
    if "embedContent" in m.supported_generation_methods:
        print(m.name)

/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [9]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vectorstore = Chroma.from_documents(
    chunks,
    embeddings,
    persist_directory=PERSIST_DIR
)

print("Vector store siap.")

Vector store siap.


## 8. Susun chain tanya-jawab (LCEL)

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

prompt = ChatPromptTemplate.from_template("""
Jawab pertanyaan HANYA berdasarkan konteks berikut. Sertakan judul sumber di akhir jawaban.
Kalau jawabannya tidak ada di konteks, katakan tidak ditemukan.

Konteks:
{context}

Pertanyaan: {question}
""")

def format_docs(docs):
    return "\n\n".join(f"[{d.metadata.get('source_title')}] {d.page_content}" for d in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## 9. Uji coba

Ganti pertanyaan di bawah sesuai isi paper kamu, jalankan beberapa kali dengan pertanyaan berbeda untuk mengecek akurasi retrieval.

In [11]:
response = chain.invoke("Metode apa yang digunakan untuk ekstraksi fitur dalam penelitian ini?")
print(response)

Metode yang digunakan untuk ekstraksi fitur dalam penelitian ini adalah model pretrained ResNet50.

Sumber: [Machine Learning Algorithms for Eye Disease Classification]


In [12]:
# Cek dokumen mentah yang terambil, untuk memastikan retrieval mengambil chunk yang relevan
retrieved_docs = retriever.invoke("Metode apa yang digunakan untuk ekstraksi fitur dalam penelitian ini?")
for i, doc in enumerate(retrieved_docs):
    print(f" Chunk {i+1} ")
    print(doc.page_content[:300])
    print()

 Chunk 1 
mengubah citra dua dimensi menjadi vektor fitur berdimensi 
2048 yang mengandung informasi penting terkait tekstur, 
pola, dan karakteristik visual retina.  Hasil ekstraksi dataset 
disimpan dalam format file  csv. Hasil ini disimpan untuk 
analisis dan klasifikasi lebih lanjut. Hasil ekstraksi fitu

 Chunk 2 
efisiensi komputasi dan mencegah overvitting , Data hasil 
ekstraksi kemudian dinormalisasi sebelum digunakan dalam 
proses klasifikasi menggunakan algoritma machine learning. 
Tujuan penelitian adalah meningkatkan akurasi klasifikasi 
penyakit mata menggunakan metode machine lear ning dan 
mengiden

 Chunk 3 
e-ISSN: 2548-6861  
JAIC Vol. 9, No. 4, August 2025:  1835 – 1845 
1836 
lebih mudah dipahami cara kerjanya dan masing-masing fitur 
terhadap hasil klasifikasi. [11]. 
Penelitian ini menggunakan dataset gamba r retina yang 
berisi empat kelas  kondisi mata  yaitu Normal, Diabetic 
Retinopathy, Glauc

 Chunk 4 
melalui beberapa tahap yang melibatkan ekstraksi fitur

## 10. Pertanyaan uji tambahan (opsional)

Tempat mencoba beberapa pertanyaan lain sekaligus, termasuk satu pertanyaan yang sengaja di luar isi paper untuk memastikan sistem jujur mengatakan tidak ditemukan (uji anti-halusinasi).

In [13]:
test_questions = [
    "Berapa akurasi model yang dilaporkan dalam penelitian ini?",
    "Dataset apa yang digunakan dalam penelitian ini?",
    "Apa hasil penelitian tentang pengobatan kanker paru-paru?"  # sengaja di luar topik
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {chain.invoke(q)}")
    print("-" * 60)

Q: Berapa akurasi model yang dilaporkan dalam penelitian ini?
A: Akurasi model yang dilaporkan dalam penelitian ini adalah sebagai berikut:

**Dari TABEL 3 HASIL PERBANDINGAN MODEL:**
*   Logistic Regression: 90.74%
*   Decission Tree: 78.76%
*   Random Forest: 89.30%
*   Gradient Boosting: 91.27%
*   Support Vector Machine: 90.74%
*   AdaBoost: 77.47%
*   Naive Bayes: 75.42%
*   K-Nearest Neighbors: 87.71%
*   Multi Layer Perceptron: 91.50%
*   XGBoost: 92.03%
*   LightGBM: 91.88%

**Dari Model 10-Fold Cross Validation:**
*   Logistic Regression: 91.42%
*   Decission Tree: 77.87%
*   Random Forest: 88.73%

**Rata-rata akurasi dari teks:**
*   Multi Layer Perceptron (MLP): 91.94%

[Machine Learning Algorithms for Eye Disease Classification]
------------------------------------------------------------
Q: Dataset apa yang digunakan dalam penelitian ini?
A: Dataset yang digunakan dalam penelitian ini adalah Eye Diseases Dataset, yang diperoleh dari platform Kaggle.

[Machine Learning Algo